# Laboratorio 01
## capa bronze

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

spark = (
    SparkSession.builder
    .appName("MusicDiscoveryLab-01")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

BASE_PATH = Path("/home/jovyan/data")

PATHS = {
    "raw": BASE_PATH / "raw",
    "bronze": BASE_PATH / "bronze" / "songs",
    "silver": BASE_PATH / "silver" / "songs",
    "gold": BASE_PATH / "gold" / "genre_metrics",
    "quarantine": BASE_PATH / "quarantine" / "songs",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Directorio de datos:", BASE_PATH)

Spark: 4.2.0
Master: local[*]
Directorio de datos: /home/jovyan/data


### Celda 2. Generar datos sintéticos reproducibles
Incluiremos registros válidos, un BPM fuera de rango, una canción sin título y un registro duplicado. Así podremos comprobar que las reglas de calidad funcionan.
Este archivo se genera siempre con los mismos datos. No depende de una API ni de un generador aleatorio.

In [4]:
import csv

columns = [
    "source_row_id",
    "song_id",
    "title",
    "artist",
    "genre",
    "bpm",
    "energy",
    "valence",
    "instrumentalness",
]

songs = [
    ["001", "S001", "Moonlit Waltz", "Artist A", "Classical", "92",  "0.42", "0.30", "0.98"],
    ["002", "S002", "Dark Ballroom", "Artist B", "Classical", "108", "0.66", "0.24", "0.95"],
    ["003", "S003", "Silent Forest", "Artist C", "Ambient",   "70",  "0.25", "0.18", "1.00"],
    ["004", "S004", "Midnight Rain", "Artist D", "Ambient",   "80",  "0.32", "0.35", "0.90"],
    ["005", "S005", "Velvet Shadows", "Artist E", "Soundtrack", "115", "0.58", "0.28", "0.96"],
    ["006", "S006", "Silver Strings", "Artist F", "Soundtrack", "100", "0.48", "0.45", "0.88"],
    ["007", "S007", "Crimson Waltz", "Artist G", "Classical", "130", "0.72", "0.40", "0.92"],
    ["008", "S008", "Echoes of Winter", "Artist H", "Ambient", "65", "0.20", "0.15", "0.99"],
    ["009", "S009", "Clockwork Dreams", "Artist I", "Soundtrack", "124", "0.75", "0.55", "0.82"],
    ["010", "S010", "", "Artist J", "Ambient", "75", "0.30", "0.20", "0.95"],
    ["011", "S011", "Broken Tempo", "Artist K", "Soundtrack", "350", "0.50", "0.40", "0.85"],
    ["012", "S012", "Velvet Shadows", "Artist E", "Soundtrack", "115", "0.58", "0.28", "0.96"],
]

raw_file = PATHS["raw"] / "songs.csv"

with raw_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(songs)

print(f"Archivo generado: {raw_file}")
print(f"Registros de entrada: {len(songs)}")

Archivo generado: /home/jovyan/data/raw/songs.csv
Registros de entrada: 12
